# Exercises XP Gold - RNNs in PyTorch on MNIST CSV

This is a guided notebook for the exercise on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear for key ideas.


## What you will learn
- Implement and train RNN, GRU, and LSTM with PyTorch
- Preprocess and load sequence datasets
- Evaluate models with accuracy
- Use Dataset and DataLoader for efficient IO


## What you will create
- A custom dataset class for MNIST CSV
- Three sequence models: SimpleRNN, SimpleGRU, SimpleLSTM
- A training loop and an evaluation function


# Part 1: Setting up the environment

**As stated in the exercise**  
Install libraries, define hyperparameters, and set the device.

**PREFILLED**  
Imports, seed control, hyperparameters, device selection, and simple helpers.


In [1]:
# PREFILLED: just execute
import os, time, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters - you may adjust in the To-Do cell below
INPUT_SIZE   = 28   # number of features per time step for MNIST row-wise
SEQ_LEN      = 28   # number of time steps
HIDDEN_SIZE  = 128
NUM_LAYERS   = 1
NUM_CLASSES  = 10
LR           = 1e-3
EPOCHS       = 5
BATCH_SIZE   = 128

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

**To-Do (written):** Briefly justify the choice of treating a 28x28 image as a sequence of length 28 with input size 28. When would column-wise sequencing change anything in practice?

Nous traitons une image MNIST de taille 28×28 comme une séquence de longueur 28 avec une taille d’entrée de 28 en considérant chaque ligne de l’image comme un pas de temps et les 28 pixels de cette ligne comme les caractéristiques d’entrée. Cette représentation permet à un RNN ou un LSTM de parcourir progressivement l’image et d’apprendre les relations entre les différentes lignes afin de reconnaître le chiffre. Un séquençage colonne par colonne fournirait les mêmes informations mais dans un ordre différent. Dans le cas de MNIST, cela a généralement peu d’impact sur les performances car les chiffres sont simples et relativement symétriques, mais pour des données où l’information est davantage structurée dans une direction particulière, le choix entre un parcours par lignes ou par colonnes peut influencer les résultats du modèle.

# Part 2: Designing the neural network models

**As stated in the exercise**  
Implement SimpleRNN, SimpleGRU, and SimpleLSTM. Each ends with a fully connected layer mapping the last hidden state to 10 classes.

**PREFILLED**  
Model skeletons with clear To-Do regions.


In [11]:
# PREFILLED: just execute
class SimpleRNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # To-Do: define self.rnn using nn.RNN with batch_first=True
        # To-Do: define self.fc as nn.Linear(hidden_size, num_classes)
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: [batch, seq_len, input_size]
        # To-Do: run the RNN, get the last hidden state, pass through self.fc
        out, _ = self.rnn(x)
        # Return logits [batch, num_classes]
        last_hidden_state = out[:, -1, :]
        logits = self.fc(last_hidden_state)
        return logits


class SimpleGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # To-Do: define self.gru using nn.GRU with batch_first=True
        # To-Do: define self.fc as nn.Linear(hidden_size, num_classes)
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # To-Do: run GRU and map last state to logits
        out, _ = self.gru(x)
        logits = self.fc(out[:, -1, :])
        return logits


class SimpleLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # To-Do: define self.lstm using nn.LSTM with batch_first=True
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # To-Do: define self.fc as nn.Linear(hidden_size, num_classes)
        self.fc   = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # To-Do: run LSTM, use hidden state h_n from the last layer, map to logits
        _, (h_n, c_n) = self.lstm(x)
        logits = self.fc(h_n[-1])  # état caché final de la dernière couche
        return logits

**Learning point**  
RNN keeps one hidden. GRU adds gates that control update and reset. LSTM keeps a cell state with input, forget, and output gates. These gates help retain or discard information over time steps.


# Part 3: Creating a custom dataset class

**As stated in the exercise**  
Create `MnistCsvDataset` that loads the CSV, preprocesses, and yields items compatible with sequences.

Dataset format expectation: a CSV with `label` as the first column, followed by 784 pixel columns named `pixel0 ... pixel783` or unnamed numeric columns. If your CSV differs, adapt the loader.


In [7]:
class MnistCsvDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        df = pd.read_csv(csv_path)
        # To-Do: extract labels as a 1D numpy array of dtype np.int64 under variable self.y
        # To-Do: extract pixel values into 2D array of shape [N, 784], type np.float32 in [0,1]
        # To-Do: reshape each row to [28, 28] and store as self.X (do not transpose)
        # Hints: scale by 255.0 if values are 0..255
        self.y = df.iloc[:, 0].values.astype(np.int64)
        self.X = df.iloc[:, 1:].values.astype(np.float32) / 255.0

    def __len__(self):
        # To-Do: return dataset length
        return len(self.y)

    def __getitem__(self, idx):
        # To-Do: return a tuple (seq, label) where
        #   seq is a torch.FloatTensor of shape [SEQ_LEN, INPUT_SIZE]
        #   label is a torch.LongTensor scalar
        seq = torch.FloatTensor(self.X[idx].reshape(SEQ_LEN, INPUT_SIZE))
        label = torch.LongTensor([self.y[idx]])[0] # Ensure scalar
        return seq, label

**To-Do (code):** Instantiate training and test datasets from CSV files. Wrap them in DataLoaders.

Use variables below. Replace the paths with your local paths.


In [8]:
# To-Do: create datasets and loaders
DATA_DIR = Path("/content/sample_data/")  # change this
TRAIN_CSV = DATA_DIR / "mnist_train_small.csv"
TEST_CSV  = DATA_DIR / "mnist_test.csv"
ds_train = MnistCsvDataset(TRAIN_CSV)
ds_test  = MnistCsvDataset(TEST_CSV)
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)

# Part 4: Training the model

**As stated in the exercise**  
Use CrossEntropyLoss and Adam. Implement a training loop that prints epoch loss.


In [9]:
# PREFILLED: training helpers
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_batches = 0
    for xb, yb in loader:
        # To-Do: move xb, yb to device and ensure xb shape [B, SEQ_LEN, INPUT_SIZE]
        xb = xb.to(device)
        yb = yb.to(device)
        # To-Do: forward, compute loss, backward, step, zero_grad
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # Accumulate total_loss
        total_loss += loss.item()
        total_batches += 1
    return total_loss / max(1, total_batches)

def fit(model, dl_train, dl_val, epochs, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_val = math.inf
    for epoch in range(1, epochs+1):
        t0 = time.time()
        train_loss = train_one_epoch(model, dl_train, criterion, optimizer, device)
        val_loss = evaluate_loss(model, dl_val, criterion, device)
        dt = time.time() - t0
        print(f"epoch {epoch:02d} - train_loss {train_loss:.4f} - val_loss {val_loss:.4f} - time_s {dt:.1f}")
        if val_loss < best_val:
            best_val = val_loss
    return model


**To-Do (code):** Implement `evaluate_loss` to compute average loss on a loader. Then construct one of the models, send to device, and call `fit`.


In [12]:
# To-Do: evaluation and training
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total = 0.0; n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total += loss.item() * xb.size(0)
            n += xb.size(0)
    return total / max(1, n)

rnn_model = SimpleRNNModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
rnn_model = fit(rnn_model, dl_train, dl_test, EPOCHS, device)

epoch 01 - train_loss 1.2978 - val_loss 0.8272 - time_s 6.0
epoch 02 - train_loss 0.7368 - val_loss 0.6341 - time_s 6.3
epoch 03 - train_loss 0.5931 - val_loss 0.5195 - time_s 6.2
epoch 04 - train_loss 0.4856 - val_loss 0.4369 - time_s 6.4
epoch 05 - train_loss 0.3982 - val_loss 0.3828 - time_s 6.8


# Part 5: Evaluating the model

**As stated in the exercise**  
Implement accuracy for train and test. Use the trained model to predict test labels and compute accuracy.


In [13]:
# To-Do: accuracy function
def accuracy(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(1, total)

print({'train_acc': accuracy(rnn_model, dl_train, device), 'test_acc': accuracy(rnn_model, dl_test, device)})

{'train_acc': 0.8947947397369869, 'test_acc': 0.8896889688968896}


# To-Do (code): Repeat training and evaluation for `SimpleGRUModel` and `SimpleLSTMModel`. Compare the final test accuracies.


In [15]:

print("\n GRU  ")
gru_model = SimpleGRUModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
gru_model = fit(gru_model, dl_train, dl_test, EPOCHS, device)
print({'gru_train_acc': accuracy(gru_model, dl_train, device), 'gru_test_acc': accuracy(gru_model, dl_test, device)})

print("\n LSTM ")
lstm_model = SimpleLSTMModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
lstm_model = fit(lstm_model, dl_train, dl_test, EPOCHS, device)
print({'lstm_train_acc': accuracy(lstm_model, dl_train, device), 'lstm_test_acc': accuracy(lstm_model, dl_test, device)})


 GRU  
epoch 01 - train_loss 1.2893 - val_loss 0.7173 - time_s 13.9
epoch 02 - train_loss 0.5322 - val_loss 0.4429 - time_s 14.2
epoch 03 - train_loss 0.3492 - val_loss 0.3077 - time_s 14.8
epoch 04 - train_loss 0.2492 - val_loss 0.2386 - time_s 15.0
epoch 05 - train_loss 0.2043 - val_loss 0.2076 - time_s 14.6
{'gru_train_acc': 0.9409970498524927, 'gru_test_acc': 0.9403940394039404}

 LSTM 
epoch 01 - train_loss 1.1993 - val_loss 0.5807 - time_s 8.2
epoch 02 - train_loss 0.4180 - val_loss 0.3445 - time_s 8.2
epoch 03 - train_loss 0.2860 - val_loss 0.2633 - time_s 8.2
epoch 04 - train_loss 0.2143 - val_loss 0.1996 - time_s 7.9
epoch 05 - train_loss 0.1784 - val_loss 0.1716 - time_s 8.0
{'lstm_train_acc': 0.9558977948897445, 'lstm_test_acc': 0.947994799479948}
